If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec06_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 6: Table Review and the Census
v.ekc

Today is a workout day: we review every table move you've learned on two real datasets — our own class survey, and the U.S. Census. (Slides: KL Lecture 06 deck.)

**Today's sections:**
1. Reading a table from a file — Du Bois, continued
2. Table review — welcome survey
3. Census data
4. 2019 sex ratios and our first line plot

In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

---
## 1. Reading a Table from a File — Du Bois, continued

Picking up the Du Bois dataset from last time (KL deck, slide 5) — watch how arithmetic on columns builds new columns.

In [ ]:
du_bois = Table.read_table('du_bois.csv')
du_bois

In [ ]:
du_bois.column('ACTUAL AVERAGE')

In [ ]:
du_bois.column('FOOD')

In [ ]:
food_dollars = du_bois.column('ACTUAL AVERAGE') * du_bois.column('FOOD')
food_dollars

In [ ]:
du_bois = du_bois.with_column('Food $', food_dollars)
du_bois

In [ ]:
np.max(du_bois.column('Food $'))

In [ ]:
du_bois.with_column(
    'Fraction of well-to-do food $', 
    du_bois.column('Food $') / np.max(du_bois.column('Food $')))

### Try It 1: Rent, revisited 😊

1. Use table methods to find the income bracket (`CLASS`) that spent the **highest proportion of income on rent**. (Same question as last lecture — can you do it without peeking?)

In [ ]:
# 1. class with highest rent proportion


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Sort, then grab the top — `.item(0)` pulls the answer out as a value.*

```python
# 1.
du_bois.sort('RENT', descending=True).column('CLASS').item(0)
```

</details>

---
## 2. Table Review — Welcome Survey

Now the fun one: **your** data. This is the welcome survey a previous class filled out. Let's put every table operation to work.

### Try It 2: Survey says… 😊

0. In your current folder there's a file called `data111_survey_fa24.csv`. Read it into a table named `welcome`.

1. What is the largest number of piercings someone has?

2. On average, how long do side-sleepers sleep?

3. What *proportion* of students are back-sleepers?

4. How many students get at least 8 hours of sleep each night? Can you find **two different ways**?

5. Create a table with only the two sleep-related columns, named `Hours` and `Position`.

In [ ]:
# 0. read the survey into `welcome`


Checkpoint — run this cell so the remaining prompts (and the rest of the lecture) work:

In [ ]:
# run me: needed below
welcome = Table.read_table('data111_survey_fa24.csv')
welcome.show(5)

In [ ]:
# 1. most piercings


In [ ]:
side_sleepers = welcome.where('Sleep Position', are.containing('side'))
side_sleepers.show(8)
...

In [ ]:
# 3. proportion of back-sleepers


In [ ]:
# First way:


In [ ]:
# Second way


In [ ]:
# 5. sleep columns, renamed


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Every one of these is a chain of moves you already know: `where`, `column`, `num_rows`, `select`, `relabeled`.*

```python
# 0.
welcome = Table.read_table('data111_survey_fa24.csv')

# 1.
max(welcome.column('Piercings'))

# 2.
side_sleepers = welcome.where('Sleep Position', are.containing('side'))
np.average(side_sleepers.column('Hours of Sleep'))

# 3.
welcome.where('Sleep Position', are.containing('back')).num_rows / welcome.num_rows

# 4. first way
welcome.where('Hours of Sleep', are.above_or_equal_to(8)).num_rows
# 4. second way
np.count_nonzero(welcome.column('Hours of Sleep') >= 8)

# 5.
(welcome.select('Hours of Sleep', 'Sleep Position')
        .relabeled('Hours of Sleep', 'Hours')
        .relabeled('Sleep Position', 'Position'))
```

</details>

---
## 3. Census Data 🇺🇸

Every 10 years the U.S. counts *everyone* — and publishes yearly estimates in between. Real datasets come with **codes** you have to decode (KL deck, slides 13–15):

> **Reading this table:** in `SEX`, 0 = total, 1 = male, 2 = female. In `AGE`, 999 means "all ages combined." Columns like `POPESTIMATE2014` are estimates for that year. Always find the data dictionary before you analyze!

In [ ]:
full = Table.read_table('nc-est2019-agesex-res.csv')
full

In [ ]:
partial = full.select('SEX', 'AGE', 'POPESTIMATE2014', 'POPESTIMATE2019')
partial.show(5)

In [ ]:
us_pop = partial.relabeled(2, '2014').relabeled(3, '2019')
us_pop.show(5)

In [ ]:
us_pop.where('AGE', are.above_or_equal_to(100)).sort('AGE')

---
## 4. 2019 Sex Ratios

Let's compare female and male populations at every age. New moves: `are.not_equal_to` to drop the totals, and dividing one column by another.

### 📋 Board Reference

| Code | What it does |
|---|---|
| `t.relabeled(2, '2014')` | rename a column **by position** |
| `t.where('AGE', are.equal_to(999))` | keep the "all ages" rows |
| `t.where('AGE', are.not_equal_to(999))` | drop them |
| `arr_a / arr_b` | elementwise ratio of two arrays |
| `t.plot('x_col', 'y_col')` | line plot |

In [ ]:
us_pop_2019 = us_pop.drop('2014')
us_pop_2019.show(3)

In [ ]:
all_ages = us_pop_2019.where('AGE', are.equal_to(999))
all_ages

In [ ]:
infants = us_pop_2019.where('AGE', are.equal_to(0))
infants

In [ ]:
females_all_rows = us_pop_2019.where('SEX', are.equal_to(2))
females = females_all_rows.where('AGE', are.not_equal_to(999))
females.show(3)

In [ ]:
males_all_rows = us_pop_2019.where('SEX', are.equal_to(1))
males = males_all_rows.where('AGE', are.not_equal_to(999))
males.show(3)

In [ ]:
f_to_m_ratios = females.column(2) / males.column(2)

ratios = Table().with_columns(
    'Age', females.column('AGE'),
    'F:M Ratio', f_to_m_ratios
)

ratios

In [ ]:
ratios.sort('Age', descending=True)

---
## 5. Our First Line Plot (of many)

Numbers in a sorted table hint at the story; the plot *shows* it.

In [ ]:
ratios.plot('Age', 'F:M Ratio')

> **What do you see?** Near age 0 the ratio is just below 1 (slightly more baby boys are born), it hovers near 1 through mid-life, then climbs steeply after 75 — women tend to live longer. One line plot, three demographic stories. 📈

---
> **Today's takeaway:** review done — you can now read, clean, filter, and combine columns on real data. Homework 3 leans on exactly these moves.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — ratio of two groups
```python
a = t.where('group', 1).column('value')
b = t.where('group', 2).column('value')
Table().with_columns('x', xs, 'ratio', b / a).plot('x', 'ratio')
```